# Tutorial 5: Population Genetics Export and Integration

This tutorial demonstrates how to export BOLDGenotyper results to formats compatible with population genetics software.

## Learning Objectives

By the end of this tutorial, you will:
- Understand population genetics export formats
- Export data for Arlequin, PopART, and DnaSP
- Prepare data for downstream population genetics analysis
- Integrate BOLDGenotyper with phylogeographic workflows
- Generate haplotype networks and diversity statistics

## Prerequisites

- Completed Tutorial 1 (Basic Genotyping Workflow)
- Understanding of population genetics concepts
- Dataset with geographic structure: `Sphyrna_lewini_scallopedhammerhead.tsv`

## Supported Export Formats

BOLDGenotyper can export to:
1. **Arlequin** (.arp) - AMOVA, genetic diversity, population structure
2. **PopART** (.nex, .csv) - Haplotype networks, TCS analysis
3. **DnaSP** (.fas) - Sequence diversity, neutrality tests
4. **Generic** (.csv, .fas) - For other software

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

## Step 1: Run Genotyping Analysis with Population Groupings

First, ensure you have genotype assignments with geographic classifications.

In [ ]:
# Run analysis with geographic assignment
!boldgenotyper ../data/Sphyrna_lewini_scallopedhammerhead.tsv \
  --clustering-threshold 0.02 \
  --build-tree \
  --output ../data/Sphyrna_lewini_popgen/ \
  --threads 4

## Step 2: Export to All Formats

Export the results to all supported population genetics formats.

In [ ]:
# Export to all formats
# The --export-format all option generates all population genetics exports

!boldgenotyper ../data/Sphyrna_lewini_scallopedhammerhead.tsv \
  --clustering-threshold 0.02 \
  --export-format all \
  --output ../data/Sphyrna_lewini_popgen/ \
  --threads 4

## Step 3: Examine Export Directory Structure

In [ ]:
# List export directory
export_dir = Path("../data/Sphyrna_lewini_popgen/exports")

if export_dir.exists():
    print("Export directory structure:")
    !tree ../data/Sphyrna_lewini_popgen/exports/
else:
    print(f"Export directory not found at: {export_dir}")

### Expected directory structure:
```
exports/
├── README.md                    # Export format documentation
├── arlequin/
│   ├── by_ocean_basin.arp      # Arlequin project file
│   └── by_country.arp
├── popart/
│   ├── haplotypes.nex          # Nexus format for PopART
│   ├── haplotypes.csv          # Frequency table
│   └── traits.csv              # Geographic metadata
├── dnasp/
│   ├── all_sequences.fas       # FASTA for DnaSP
│   └── by_ocean_basin.fas
└── generic/
    ├── genotype_sequences.fas  # Consensus sequences
    ├── haplotype_table.csv     # Genotype frequencies
    └── metadata.csv            # Full metadata
```

## Step 4: Analyze Arlequin Export

Arlequin format is used for AMOVA, Fst, and genetic diversity analyses.

In [ ]:
# Read Arlequin file (first 50 lines)
arlequin_file = export_dir / "arlequin" / "by_ocean_basin.arp"

if arlequin_file.exists():
    print("Arlequin project file (first 50 lines):")
    print("="*80)
    with open(arlequin_file, 'r') as f:
        for i, line in enumerate(f):
            if i < 50:
                print(line.rstrip())
            else:
                break
    print("="*80)
    print(f"\nFull file: {arlequin_file}")
    print("\nOpen in Arlequin 3.5 to perform:")
    print("  - AMOVA (ocean basin structure)")
    print("  - Pairwise Fst calculations")
    print("  - Nucleotide diversity")
    print("  - Mismatch distributions")
else:
    print(f"Arlequin file not found at: {arlequin_file}")

## Step 5: Analyze PopART Export

PopART format is used for haplotype network construction.

In [ ]:
# Load PopART haplotype frequency table
popart_csv = export_dir / "popart" / "haplotypes.csv"

if popart_csv.exists():
    haplotypes = pd.read_csv(popart_csv)
    print("PopART Haplotype Frequency Table:")
    print(haplotypes.head(20))
    
    print(f"\nTotal haplotypes: {len(haplotypes)}")
    print(f"Total samples: {haplotypes.iloc[:, 1:].sum().sum()}")
    
    # Show traits file
    traits_file = export_dir / "popart" / "traits.csv"
    if traits_file.exists():
        traits = pd.read_csv(traits_file)
        print(f"\nTraits file (geographic metadata):")
        print(traits.head(10))
else:
    print(f"PopART file not found at: {popart_csv}")

In [ ]:
# Visualize haplotype frequency distribution
if popart_csv.exists():
    # Calculate total frequency for each haplotype
    haplotypes['total_freq'] = haplotypes.iloc[:, 1:].sum(axis=1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Frequency distribution
    axes[0].hist(haplotypes['total_freq'], bins=30, edgecolor='black')
    axes[0].set_xlabel('Haplotype Frequency')
    axes[0].set_ylabel('Number of Haplotypes')
    axes[0].set_title('Haplotype Frequency Distribution')
    axes[0].set_yscale('log')
    
    # Top 20 haplotypes
    top20 = haplotypes.nlargest(20, 'total_freq')
    axes[1].barh(range(len(top20)), top20['total_freq'])
    axes[1].set_yticks(range(len(top20)))
    axes[1].set_yticklabels([f"H{i+1}" for i in range(len(top20))])
    axes[1].set_xlabel('Frequency')
    axes[1].set_ylabel('Haplotype')
    axes[1].set_title('Top 20 Most Common Haplotypes')
    
    plt.tight_layout()
    plt.show()

## Step 6: Analyze DnaSP Export

DnaSP format contains aligned sequences for diversity analysis.

In [ ]:
# Count sequences in DnaSP export
dnasp_file = export_dir / "dnasp" / "all_sequences.fas"

if dnasp_file.exists():
    # Count sequences
    with open(dnasp_file, 'r') as f:
        content = f.read()
        n_seqs = content.count('>')
    
    print(f"DnaSP FASTA file: {dnasp_file}")
    print(f"Total sequences: {n_seqs}")
    
    # Show first few sequences
    print("\nFirst 20 lines:")
    print("="*80)
    with open(dnasp_file, 'r') as f:
        for i, line in enumerate(f):
            if i < 20:
                print(line.rstrip())
            else:
                break
    print("="*80)
    
    print("\nOpen in DnaSP to calculate:")
    print("  - Nucleotide diversity (π)")
    print("  - Haplotype diversity (Hd)")
    print("  - Tajima's D")
    print("  - Fu's Fs")
else:
    print(f"DnaSP file not found at: {dnasp_file}")

## Step 7: Calculate Basic Diversity Metrics

Calculate some basic diversity metrics from the genotype data.

In [ ]:
# Load annotated results
results = pd.read_csv("../data/Sphyrna_lewini_popgen/Sphyrna_lewini_annotated.csv")

# Overall genotype diversity
n_genotypes = results['genotype'].nunique()
n_samples = len(results)

# Calculate haplotype diversity (Nei 1987)
genotype_freqs = results['genotype'].value_counts(normalize=True)
hd = (n_samples / (n_samples - 1)) * (1 - (genotype_freqs ** 2).sum())

print("OVERALL GENETIC DIVERSITY")
print("="*80)
print(f"Sample size: {n_samples}")
print(f"Number of genotypes: {n_genotypes}")
print(f"Haplotype diversity (Hd): {hd:.4f}")
print(f"Genotypes per sample: {n_genotypes/n_samples:.4f}")
print("="*80)

In [ ]:
# Diversity by ocean basin
if 'ocean_basin' in results.columns:
    print("\nGENETIC DIVERSITY BY OCEAN BASIN")
    print("="*80)
    
    diversity_by_basin = []
    
    for basin in results['ocean_basin'].dropna().unique():
        basin_data = results[results['ocean_basin'] == basin]
        n = len(basin_data)
        n_geno = basin_data['genotype'].nunique()
        
        # Haplotype diversity
        freqs = basin_data['genotype'].value_counts(normalize=True)
        hd_basin = (n / (n - 1)) * (1 - (freqs ** 2).sum()) if n > 1 else 0
        
        diversity_by_basin.append({
            'basin': basin,
            'n_samples': n,
            'n_genotypes': n_geno,
            'haplotype_diversity': hd_basin
        })
    
    diversity_df = pd.DataFrame(diversity_by_basin)
    diversity_df = diversity_df.sort_values('n_samples', ascending=False)
    
    print(diversity_df.to_string(index=False))
    print("="*80)

In [ ]:
# Visualize diversity metrics
if 'ocean_basin' in results.columns and len(diversity_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Sample size vs genotype count
    axes[0].scatter(diversity_df['n_samples'], diversity_df['n_genotypes'], s=100)
    for idx, row in diversity_df.iterrows():
        axes[0].text(row['n_samples'], row['n_genotypes'], 
                    row['basin'][:10], fontsize=8)
    axes[0].set_xlabel('Sample Size')
    axes[0].set_ylabel('Number of Genotypes')
    axes[0].set_title('Genotype Richness by Basin')
    axes[0].grid(True, alpha=0.3)
    
    # Haplotype diversity by basin
    diversity_df_sorted = diversity_df.sort_values('haplotype_diversity', ascending=True)
    axes[1].barh(range(len(diversity_df_sorted)), 
                diversity_df_sorted['haplotype_diversity'])
    axes[1].set_yticks(range(len(diversity_df_sorted)))
    axes[1].set_yticklabels([b[:20] for b in diversity_df_sorted['basin']])
    axes[1].set_xlabel('Haplotype Diversity (Hd)')
    axes[1].set_ylabel('Ocean Basin')
    axes[1].set_title('Genetic Diversity by Ocean Basin')
    axes[1].set_xlim(0, 1)
    
    plt.tight_layout()
    plt.show()

## Step 8: Example PopART Workflow

Instructions for creating haplotype networks in PopART.

In [ ]:
# Print PopART workflow
popart_workflow = """
POPART HAPLOTYPE NETWORK WORKFLOW
================================================================================

1. OPEN POPART
   - Download from: http://popart.otago.ac.nz/
   - Install and launch PopART

2. IMPORT DATA
   - File > Open
   - Select: exports/popart/haplotypes.nex
   - Import traits: exports/popart/traits.csv

3. CONSTRUCT NETWORK
   - Network > TCS Network (recommended)
   - Alternative: Median-Joining Network
   - Set connection limit: 95% (default)

4. CUSTOMIZE VISUALIZATION
   - Color by: ocean_basin (or other trait)
   - Size by: frequency
   - Show trait pie charts for shared haplotypes

5. EXPORT FIGURE
   - File > Export > SVG/PNG
   - Publication-quality vector graphics

6. INTERPRET RESULTS
   - Star-like pattern: Recent expansion
   - Geographic structure: Limited gene flow
   - Deep splits: Ancient divergence

================================================================================
"""

print(popart_workflow)

## Step 9: Example Arlequin Workflow

Instructions for population structure analysis in Arlequin.

In [ ]:
# Print Arlequin workflow
arlequin_workflow = """
ARLEQUIN AMOVA WORKFLOW
================================================================================

1. OPEN ARLEQUIN
   - Download from: http://cmpg.unibe.ch/software/arlequin35/
   - Install Arlequin 3.5

2. IMPORT PROJECT
   - File > Open Project
   - Select: exports/arlequin/by_ocean_basin.arp

3. CONFIGURE AMOVA
   - AMOVA > Settings
   - Number of permutations: 10000
   - Distance method: Pairwise differences
   - Check: Compute pairwise FST

4. RUN ANALYSIS
   - AMOVA > Start
   - Wait for completion (may take several minutes)

5. VIEW RESULTS
   - Results window shows:
     * Variance components
     * Fixation indices (FST, FCT, FSC)
     * P-values from permutation test

6. PAIRWISE FST
   - View pairwise FST matrix
   - Identify most differentiated populations
   - Export matrix for visualization

7. ADDITIONAL ANALYSES
   - Nucleotide diversity (π)
   - Mismatch distribution
   - Neutrality tests (if applicable)

INTERPRETATION:
- FST < 0.05: Low differentiation
- FST 0.05-0.15: Moderate differentiation  
- FST 0.15-0.25: High differentiation
- FST > 0.25: Very high differentiation

================================================================================
"""

print(arlequin_workflow)

## Step 10: Generate Publication Summary

In [ ]:
# Create publication summary
pub_summary = f"""
POPULATION GENETICS ANALYSIS SUMMARY
================================================================================

DATASET: Sphyrna lewini (scalloped hammerhead shark)
SAMPLES: {n_samples} individuals
GENOTYPES: {n_genotypes} unique COI haplotypes

GENETIC DIVERSITY:
- Overall haplotype diversity (Hd): {hd:.4f}
- Range across ocean basins: {diversity_df['haplotype_diversity'].min():.4f} - {diversity_df['haplotype_diversity'].max():.4f}

EXPORTED FORMATS:
1. Arlequin (.arp) - AMOVA and population structure
2. PopART (.nex, .csv) - Haplotype network construction
3. DnaSP (.fas) - Sequence diversity analysis

RECOMMENDED ANALYSES:
1. Haplotype network (PopART)
   - TCS network to visualize relationships
   - Color by ocean basin
   - Identify geographic structure

2. AMOVA (Arlequin)
   - Test for ocean basin differentiation
   - Calculate pairwise FST
   - Assess hierarchical structure

3. Diversity metrics (DnaSP)
   - Nucleotide diversity by region
   - Neutrality tests
   - Demographic history

EXPORT DIRECTORY: exports/
DOCUMENTATION: exports/README.md

================================================================================
Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

print(pub_summary)

# Save summary
with open("../data/Sphyrna_lewini_popgen/exports/analysis_summary.txt", 'w') as f:
    f.write(pub_summary)
    
print("\nSummary saved to: ../data/Sphyrna_lewini_popgen/exports/analysis_summary.txt")

## Key Takeaways

1. **Seamless Integration**: BOLDGenotyper exports directly to major population genetics software
2. **Multiple Formats**: Each format optimized for specific analyses
3. **Geographic Structure**: Population groupings based on ocean basins or custom classifications
4. **Publication-Ready**: Automated export with proper formatting
5. **Flexible Workflows**: Combine with other tools for comprehensive phylogeographic analysis

## Recommended Analysis Pipeline

1. **BOLDGenotyper** → Genotype assignment and geographic classification
2. **PopART** → Visualize haplotype relationships and geographic structure
3. **Arlequin** → Quantify population differentiation (AMOVA, FST)
4. **DnaSP** → Calculate diversity metrics and test demographic hypotheses
5. **BEAST/MrBayes** → Estimate divergence times (optional)
6. **R/Python** → Statistical analysis and publication figures

## Additional Software

### Haplotype Networks
- **PopART**: http://popart.otago.ac.nz/ (recommended)
- **Network**: http://www.fluxus-engineering.com/
- **TCS**: https://github.com/gjburgess/TCS

### Population Structure
- **Arlequin**: http://cmpg.unibe.ch/software/arlequin35/ (recommended)
- **STRUCTURE**: https://web.stanford.edu/group/pritchardlab/structure.html
- **ADMIXTURE**: https://dalexander.github.io/admixture/

### Diversity Analysis
- **DnaSP**: http://www.ub.edu/dnasp/ (recommended)
- **MEGA**: https://www.megasoftware.net/
- **pegas** (R): https://cran.r-project.org/package=pegas

## Best Practices

1. **Check alignment** - Verify sequences are properly aligned before export
2. **Validate groupings** - Ensure geographic classifications are biologically meaningful
3. **Document parameters** - Record clustering threshold and export settings
4. **Report sample sizes** - Include per-population sample sizes in methods
5. **Cite properly** - Acknowledge BOLDGenotyper and downstream software

## Publication Methods Template

```
COI sequences were genotyped using BOLDGenotyper v0.1.0 with a clustering
threshold of 0.02. Haplotype networks were constructed using the TCS algorithm
in PopART v1.7. Analysis of molecular variance (AMOVA) was performed in
Arlequin v3.5 to test for population structure among ocean basins (10,000
permutations). Genetic diversity indices were calculated in DnaSP v6.
```

## Additional Resources

- PopGen Export Guide: `../POPGEN_EXPORT_GUIDE.md`
- Format specifications and troubleshooting
- Advanced export options
- Integration with R and Python

## Congratulations!

You've completed all 5 BOLDGenotyper tutorials:
1. Basic genotyping workflow
2. Parameter sweep optimization  
3. Comparative analysis quality control
4. Custom shapefiles for freshwater organisms
5. Population genetics export and integration

You now have the skills to perform comprehensive COI genotyping and phylogeographic analysis for any organism with BOLD data!